# D4-0: goal-oriented adaptive time stepping (correctness and opportunity)

First reference domain of the cross-domain programme (`docs/plans/cross-domain-plan.md` §5, milestone **D4-0**, rung 0 of the ladder in §6). **Runs on CPU in a few minutes; no GPU, no Drive and no external data are needed.**

**The domain.** `y' = A y + b(t)` on `[0, 1]` (a damped oscillator coupled to one decaying or growing mode, forced by 2–4 Gaussian pulses), integrated with Crank–Nicolson on a grid the allocator refines by bisecting intervals. The quantity of interest is `cᵀ y(T)`. The declared objective is the cancellation-free goal-oriented error `Σ_j |Λ_{j+1}ᵀ τ_{j+1}|`, an exact upper bound on `|cᵀ(y(T) − y_N)|`, where `Λ` is the discrete co-state (comprehensive plan §4.2) and `τ` the local errors. `|cᵀ(y(T) − y_N)|` is reported too.

**What this notebook decides.**

1. **Correctness:** the error representation `cᵀ(y(T) − y_N) = Σ_j Λ_{j+1}ᵀ τ_{j+1}` holds, and the reference solution is converged, on the evaluation family.
2. **Opportunity (gate):** on the *validation* family, the best-known curve beats uniform refinement by at least 15 % of the normalised area over the budget range. If it doesn't, the domain is not diagnostic for allocation and D4-1 does not open.

**Reference curve.** The greedy one-step oracle is not a true oracle and can be beaten. Regret is therefore measured against the **best-known curve**: the pointwise minimum, at each budget, over every evaluated policy, privileged ones included. It upper-bounds the true oracle, so the reported regret is non-negative and a lower bound on true oracle regret. It depends on the policy set, which the report lists.
3. **Weighting comparison at equal refinement count (test family, read once):** with the *same* step-doubling local-error estimate `τ̂`, does weighting by the discrete co-state (`adjoint`) lower regret compared with no weighting (`residual`) and goal projection without propagation (`goal_local`)?
4. **Equal-compute check (validation and test):** the research plan requires comparisons at matched total compute, and scoring is not free. Every deployable policy is therefore also compared at fixed levels of *total* CN steps (re-solves after each refinement plus the cost of scoring). Deployable adaptive policies must then beat plain uniform refinement, which spends nothing on scoring and so gets many more refinements.

**Ledger.** `refine(j)` costs the CN steps that must be re-solved, from the refined interval to the end (`n + 1 − j` for `n` intervals). Scoring costs 2 steps per interval for the step-doubling estimate (`residual`, `goal_local`) and 3 for the estimate plus the backward co-state sweep (`adjoint`), at every decision. The privileged references are not charged and are excluded from the equal-compute comparison.

**Design history (disclosed).** The first execution of this notebook, at commit `ab7c7ba`, charged only 1 step per refinement and compared policies at equal refinement count only. A ledger audit then found the under-charge, and a validation-family preview (the test family was not used) showed uniform refinement ahead of every adaptive policy at equal compute. The corrected ledger, the equal-compute check and the rule below were added after that preview. The compute levels were fixed from ledger arithmetic on validation, not tuned. The refinement-count results do not depend on the ledger (no policy here reads costs), and the notebook's README records that they were identical across the two executions.

**D4-1 opens only if all three hold:** correctness passes; the rate-budget opportunity gate passes on validation; **and** on validation the co-state policy beats uniform refinement at equal compute (the CI of `mean(J_adjoint − J_uniform)` lies below 0 at two adjacent compute levels). This last condition is stricter than the plan's rate-budget gate and was added after the preview above. It is stated here, before the test family is read at equal compute.

**Claim boundary.** This is an analytic benchmark generated in the repository with frozen seeds, like `HJoinBench-0`; it is not real data. The co-state here is *computed* exactly from the known model ("Mode A"), not learned, so D4-0 says nothing about H2 (adjoint vs a realised direct critic): that is D4-1. `exact_tau_adjoint` and `one_step_oracle` use the exact solution and are diagnostics, never deployable results.

In [ ]:
# ================================================================
# 0. Package and output location (CPU is enough)
# ================================================================
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA
REPO_DIR = '/content/para_001' if IN_COLAB else str(Path.cwd().resolve().parents[1])

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)
    for candidate in (f'origin/{REPO_REF}', REPO_REF):
        resolved = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
                                  capture_output=True, text=True)
        if resolved.returncode == 0:
            subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolved.stdout.strip()], check=True)
            break
    else:
        raise RuntimeError(f'Cannot resolve {REPO_REF!r}.')
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
try:
    import adjointrwm.domains  # noqa: F401
except ImportError as error:
    raise RuntimeError(f'{REPO_REF!r} predates src/adjointrwm/domains; set REPO_REF to a commit that has it.') from error

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    RUN_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs')
else:
    RUN_ROOT = Path.home() / 'adjointrwm_runs'  # outside the repository; import results with the import-run skill
print('adjointrwm at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '', '| runs ->', RUN_ROOT)

In [ ]:
# ================================================================
# 1. Frozen configuration and immutable run directory
# ================================================================
import datetime
import json
import platform
import time
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from adjointrwm.domains import AdaptiveTimeSteppingDomain, sample_instances
from adjointrwm.io import atomic_write_json, atomic_write_text, sha256_json


@dataclass(frozen=True)
class D4Config:
    validation_seed: int = 1001     # opportunity gate and any design decision
    test_seed: int = 2002           # policy comparison, read once
    instances_per_family: int = 40
    initial_intervals: int = 16
    max_depth: int = 7              # finest grid: 16 * 2**7 = 2048 intervals
    max_steps: int = 48             # refinement budget B = 0..48
    random_draws: int = 8           # expected-random baseline
    objective_kind: str = 'bound'   # declared objective (see the markdown above)
    opportunity_min_relative: float = 0.15
    delta_oracle_min: float = 1e-8  # below this the fraction of oracle advantage is undefined
    bootstrap_resamples: int = 10000
    compute_levels: tuple = (1000.0, 2000.0, 4000.0, 8000.0)  # total CN steps; spans B ~ 10..55 for the adjoint policy on validation
    compute_max_actions: int = 2032                          # safety cap (= finest grid minus initial grid)


CFG = D4Config()
CONFIG_DICT = asdict(CFG)
CONFIG_HASH = sha256_json(CONFIG_DICT)
RUN_ID = 'd4_time_stepping_' + datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} exists; run directories are immutable.')
PATHS = {k: RUN_DIR / k for k in ('config', 'artifacts', 'figures', 'reports', 'tests')}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

DOMAIN = AdaptiveTimeSteppingDomain(objective_kind=CFG.objective_kind)
FAMILIES = {
    'validation': sample_instances(CFG.validation_seed, CFG.instances_per_family, CFG.initial_intervals, CFG.max_depth),
    'test': sample_instances(CFG.test_seed, CFG.instances_per_family, CFG.initial_intervals, CFG.max_depth),
}
INSTANCE_MANIFEST = {split: [asdict(inst) for inst in insts] for split, insts in FAMILIES.items()}
atomic_write_json(PATHS['config'] / 'run_config.json', {
    'run_id': RUN_ID, 'config': CONFIG_DICT, 'config_sha256': CONFIG_HASH, 'repo_commit': REPO_COMMIT,
    'repo_dirty': REPO_DIRTY, 'notebook': 'notebooks/04-domains/d4_adaptive_time_stepping.ipynb',
    'domain_spec': DOMAIN.spec.to_dict(), 'domain_spec_sha256': DOMAIN.spec.checksum(),
})
atomic_write_json(PATHS['config'] / 'instances.json', {**INSTANCE_MANIFEST, 'sha256': sha256_json(INSTANCE_MANIFEST)})
atomic_write_json(PATHS['config'] / 'environment.json', {
    'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'hardware': 'CPU (timings are not a systems claim)',
})
print('Run:', RUN_ID, '| config', CONFIG_HASH[:12])

In [ ]:
# ================================================================
# 2. Correctness on both families (must pass before anything else is read)
# ================================================================
from adjointrwm.domains import LinearODEInstance
from adjointrwm.domains.linear_ode import reference_solution

checks = []
for split, instances in FAMILIES.items():
    for inst in instances:
        state = DOMAIN.initial_state(inst)
        for _ in range(2):  # uniform grid and a refined, non-uniform grid
            weighted = DOMAIN.weighted_local_errors(state, inst)
            signed = DOMAIN.signed_error(state, inst)
            checks.append({'split': split, 'instance': inst.name, 'intervals': len(state.nodes) - 1,
                           'representation_abs_diff': abs(weighted.sum() - signed), 'abs_qoi_error': abs(signed)})
            state = DOMAIN.apply(state, DOMAIN.legal_candidates(state)[0])
checks = pd.DataFrame(checks)
representation_ok = bool((checks['representation_abs_diff'] <= 1e-10 * np.maximum(1.0, checks['abs_qoi_error'])).all())

# Reference convergence: the same instance on a grid twice as fine must give the same y(T).
convergence = []
for inst in FAMILIES['validation'][:5]:
    finer = LinearODEInstance(**{**inst.__dict__, 'max_depth': inst.max_depth + 1})
    diff = float(np.abs(reference_solution(inst)[-1] - reference_solution(finer)[-1]).max())
    convergence.append({'instance': inst.name, 'max_abs_diff_yT': diff})
convergence = pd.DataFrame(convergence)
reference_ok = bool((convergence['max_abs_diff_yT'] < CFG.delta_oracle_min).all())

CORRECTNESS = {'error_representation': representation_ok, 'reference_converged': reference_ok,
               'max_representation_abs_diff': float(checks['representation_abs_diff'].max()),
               'max_reference_diff': float(convergence['max_abs_diff_yT'].max()),
               'passed': representation_ok and reference_ok}
atomic_write_json(PATHS['tests'] / 'correctness.json', CORRECTNESS)
checks.to_csv(PATHS['tests'] / 'error_representation.csv', index=False)
convergence.to_csv(PATHS['tests'] / 'reference_convergence.csv', index=False)
print(json.dumps(CORRECTNESS, indent=2))
if not CORRECTNESS['passed']:
    raise RuntimeError('Correctness gate failed: nothing below may be read. See tests/.')

In [ ]:
# ================================================================
# 3. Opportunity gate on the validation family
# ================================================================
from adjointrwm.domains import (
    BEST_KNOWN, compute_level_summary, d4_policies, evaluate_at_compute, evaluate_policies, opportunity_over_budgets,
    policy_curves, with_best_known,
)

POLICIES = d4_policies()
started = time.perf_counter()
VALIDATION = evaluate_policies(DOMAIN, FAMILIES['validation'], POLICIES, CFG.max_steps, random_draws=CFG.random_draws)
print(f'validation evaluated in {time.perf_counter() - started:.0f}s')
VALIDATION.to_parquet(PATHS['artifacts'] / 'validation_curves.parquet', index=False)

curves = with_best_known(policy_curves(VALIDATION).pivot_table(index=['instance', 'budget'], columns='policy', values='objective'))
per_instance = []
for instance, block in curves.groupby(level='instance'):
    result = opportunity_over_budgets(block['uniform'].to_numpy(), block[BEST_KNOWN].to_numpy(),
                                      min_relative=CFG.opportunity_min_relative)
    per_instance.append({'instance': instance, **result})
per_instance = pd.DataFrame(per_instance)
mean_curves = curves.groupby(level='budget').mean()
aggregate = opportunity_over_budgets(mean_curves['uniform'].to_numpy(), mean_curves[BEST_KNOWN].to_numpy(),
                                     min_relative=CFG.opportunity_min_relative)
OPPORTUNITY = {**aggregate, 'fraction_of_instances_passing': float(per_instance['passes'].mean()),
               'policies_in_best_known': sorted(p.name for p in POLICIES),
               'definition': 'area(J_uniform - J_best_known) >= 0.15 * area(J_uniform) over B = 0..max_steps, validation family'}
atomic_write_json(PATHS['artifacts'] / 'opportunity_gate.json', OPPORTUNITY)
per_instance.to_csv(PATHS['artifacts'] / 'opportunity_per_instance.csv', index=False)
print(json.dumps(OPPORTUNITY, indent=2))

# Equal-compute check on validation (deployable policies only; privileged references are not charged).
started = time.perf_counter()
VAL_COMPUTE = evaluate_at_compute(DOMAIN, FAMILIES['validation'], POLICIES, CFG.compute_levels, CFG.compute_max_actions,
                                  random_draws=CFG.random_draws)
print(f'validation equal-compute evaluated in {time.perf_counter() - started:.0f}s')
VAL_COMPUTE.to_parquet(PATHS['artifacts'] / 'validation_equal_compute.parquet', index=False)
COMPUTE_PAIRS = [('adjoint', 'uniform'), ('residual', 'uniform'), ('goal_local', 'uniform'), ('adjoint', 'residual'), ('adjoint', 'goal_local')]
VAL_COMPUTE_SUMMARY = compute_level_summary(VAL_COMPUTE, COMPUTE_PAIRS, num_resamples=CFG.bootstrap_resamples)
paying = [level for level, e in VAL_COMPUTE_SUMMARY.items() if e['differences']['adjoint - uniform']['ci_high'] < 0]
levels_sorted = sorted(VAL_COMPUTE_SUMMARY)
adjacent = any(levels_sorted[i] in paying and levels_sorted[i + 1] in paying for i in range(len(levels_sorted) - 1))
EQUAL_COMPUTE_GATE = {'levels': list(CFG.compute_levels), 'levels_where_adjoint_beats_uniform': paying,
                      'adjoint_beats_uniform_at_two_adjacent_levels': bool(adjacent),
                      'rule': 'CI of mean(J_adjoint - J_uniform) below 0 at two adjacent compute levels (validation family)'}
atomic_write_json(PATHS['artifacts'] / 'equal_compute_gate.json', {**EQUAL_COMPUTE_GATE, 'summary': {str(k): v for k, v in VAL_COMPUTE_SUMMARY.items()}})
print(json.dumps(EQUAL_COMPUTE_GATE, indent=2))
for level, e in VAL_COMPUTE_SUMMARY.items():
    print(int(level), {k: round(v, 3) for k, v in e['mean_objective'].items()}, e['differences']['adjoint - uniform']['reading'])

In [ ]:
# ================================================================
# 4. Test family (read once): regret, AURC, paired differences, cost
# ================================================================
from adjointrwm.domains import aurc_table, fraction_of_oracle_advantage, paired_aurc_difference

started = time.perf_counter()
TEST = evaluate_policies(DOMAIN, FAMILIES['test'], POLICIES, CFG.max_steps, random_draws=CFG.random_draws)
print(f'test evaluated in {time.perf_counter() - started:.0f}s')
TEST.to_parquet(PATHS['artifacts'] / 'test_curves.parquet', index=False)

TABLE = aurc_table(TEST, reference=BEST_KNOWN, fixed='uniform')
TABLE.to_csv(PATHS['artifacts'] / 'test_aurc_by_instance.csv', index=False)
test_curves = with_best_known(policy_curves(TEST).pivot_table(index=['instance', 'budget'], columns='policy', values='objective'))
final = test_curves.xs(CFG.max_steps, level='budget')
SUMMARY = {}
for policy in TABLE['policy'].unique():
    rows = TABLE[TABLE['policy'] == policy]
    fractions = [fraction_of_oracle_advantage(final.loc[i, 'uniform'], final.loc[i, policy], final.loc[i, BEST_KNOWN],
                                              CFG.delta_oracle_min) for i in final.index]
    SUMMARY[policy] = {
        'mean_aurc_vs_best_known': float(rows['aurc'].mean()),
        'mean_final_regret': float(rows['final_regret'].mean()),
        'mean_adaptive_gain_area_vs_uniform': float(rows['adaptive_gain_area'].mean()),
        'median_fraction_of_oracle_advantage_at_B_max': float(np.nanmedian(fractions)) if not np.all(np.isnan(fractions)) else None,
        'undefined_fractions': int(np.isnan(fractions).sum()),
        'deployable': bool(TEST.loc[TEST['policy'] == policy, 'deployable'].iloc[0]),
        'mean_decision_compute_at_B_max': float(TEST[(TEST['policy'] == policy) & (TEST['budget'] == CFG.max_steps)]['decision_compute'].mean()),
    }

PAIRS = [('adjoint', 'residual'), ('adjoint', 'goal_local'), ('adjoint', 'uniform'), ('goal_local', 'residual'),
         ('residual', 'uniform'), ('exact_tau_adjoint', 'adjoint')]
COMPARISONS = {}
for a, b in PAIRS:
    result = paired_aurc_difference(TABLE, a, b, num_resamples=CFG.bootstrap_resamples)
    result['reading'] = (f'{a} lower AURC' if result['ci_high'] < 0 else f'{b} lower AURC' if result['ci_low'] > 0
                         else 'no detectable difference')
    COMPARISONS[f'{a} - {b}'] = result
atomic_write_json(PATHS['artifacts'] / 'test_summary.json', {'policies': SUMMARY, 'paired_aurc_differences': COMPARISONS})
print(pd.DataFrame(SUMMARY).T.to_string())
print(pd.DataFrame(COMPARISONS).T[['estimate', 'ci_low', 'ci_high', 'reading']].to_string())

# Equal-compute comparison on the test family (read once, levels frozen in the config).
started = time.perf_counter()
TEST_COMPUTE = evaluate_at_compute(DOMAIN, FAMILIES['test'], POLICIES, CFG.compute_levels, CFG.compute_max_actions,
                                   random_draws=CFG.random_draws)
print(f'test equal-compute evaluated in {time.perf_counter() - started:.0f}s')
TEST_COMPUTE.to_parquet(PATHS['artifacts'] / 'test_equal_compute.parquet', index=False)
TEST_COMPUTE_SUMMARY = compute_level_summary(TEST_COMPUTE, COMPUTE_PAIRS, num_resamples=CFG.bootstrap_resamples)
atomic_write_json(PATHS['artifacts'] / 'test_equal_compute_summary.json', {str(k): v for k, v in TEST_COMPUTE_SUMMARY.items()})
for level, e in TEST_COMPUTE_SUMMARY.items():
    print(int(level), {k: round(v, 3) for k, v in e['mean_objective'].items()}, e['differences']['adjoint - uniform']['reading'])

In [ ]:
# ================================================================
# 5. Figure (raw table first) and report
# ================================================================
import matplotlib.pyplot as plt

mean_test = test_curves.groupby(level='budget').mean()
regret = mean_test.drop(columns=BEST_KNOWN).sub(mean_test[BEST_KNOWN], axis=0)
regret.to_csv(PATHS['figures'] / 'mean_regret_by_budget.csv')
fig, ax = plt.subplots(figsize=(8, 5))
for policy in regret.columns:
    style = '--' if not SUMMARY[policy]['deployable'] else '-'
    ax.plot(regret.index, regret[policy], style, label=policy)
ax.set_xlabel('refinements (budget B)'); ax.set_ylabel('mean regret vs best-known curve'); ax.set_yscale('symlog', linthresh=1e-4)
ax.set_title(f'{RUN_ID}: test family, mean over {CFG.instances_per_family} instances (dashed = privileged)')
ax.legend()
fig.savefig(PATHS['figures'] / 'mean_regret_by_budget.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

REPORT = {
    'run_id': RUN_ID, 'milestone': 'D4-0', 'config_hash': CONFIG_HASH, 'repo_commit': REPO_COMMIT,
    'correctness': CORRECTNESS, 'opportunity_gate_validation': OPPORTUNITY,
    'equal_compute_gate_validation': EQUAL_COMPUTE_GATE,
    'test_equal_compute': {str(k): {'mean_objective': v['mean_objective'], 'mean_actions': v['mean_actions'],
                                    'differences': {n: {'estimate': d['estimate'], 'ci_low': d['ci_low'], 'ci_high': d['ci_high'], 'reading': d['reading']}
                                                    for n, d in v['differences'].items()}}
                           for k, v in TEST_COMPUTE_SUMMARY.items()},
    'd4_1_opens': bool(CORRECTNESS['passed'] and OPPORTUNITY['passes'] and EQUAL_COMPUTE_GATE['adjoint_beats_uniform_at_two_adjacent_levels']),
    'test_policies': SUMMARY, 'test_paired_aurc_differences': COMPARISONS,
    'claim_boundary': ('Analytic benchmark generated in-repo (frozen seeds). The co-state is computed exactly from the known '
                       'model (Mode A); no learned arms, so no H2 statement follows. exact_tau_adjoint and one_step_oracle '
                       'are privileged diagnostics.'),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)
lines = [f'# D4-0 `{RUN_ID}`', '', f"Correctness: **{'PASS' if CORRECTNESS['passed'] else 'FAIL'}** · "
         f"Opportunity (validation): **{'PASS' if OPPORTUNITY['passes'] else 'FAIL'}** "
         f"(relative headroom {OPPORTUNITY['relative_headroom']:.4f}, required {CFG.opportunity_min_relative}) · "
         f"Equal-compute payoff (validation): **{'PASS' if EQUAL_COMPUTE_GATE['adjoint_beats_uniform_at_two_adjacent_levels'] else 'FAIL'}** · "
         f"D4-1 opens: **{REPORT['d4_1_opens']}**", '',
         '| Paired AURC difference at equal refinement count (test) | Estimate | 95 % CI | Reading |', '|---|---:|---|---|']
for name, c in COMPARISONS.items():
    lines.append(f"| {name} | {c['estimate']:+.4g} | [{c['ci_low']:+.4g}, {c['ci_high']:+.4g}] | {c['reading']} |")
lines += ['', '| Total compute (CN steps) | Mean J adjoint | Mean J uniform | adjoint − uniform, 95 % CI | Reading |', '|---:|---:|---:|---|---|']
for level, e in TEST_COMPUTE_SUMMARY.items():
    d = e['differences']['adjoint - uniform']
    lines.append(f"| {level:g} | {e['mean_objective']['adjoint']:.4g} | {e['mean_objective']['uniform']:.4g} | "
                 f"{d['estimate']:+.4g} [{d['ci_low']:+.4g}, {d['ci_high']:+.4g}] | {d['reading']} |")
lines += ['', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
if CORRECTNESS['passed']:
    atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)

## Reading the result

| Outcome | Meaning | Next step |
|---|---|---|
| Correctness FAIL | A bug or an unconverged reference | Fix the code; nothing else in the run may be read |
| A deployable policy is the best-known curve at some budgets | Greedy one-step lookahead is myopic, so it can be beaten | Expected; it is why regret uses the best-known curve |
| Opportunity FAIL | Uniform refinement is nearly as good as the oracle on this family | Redesign the family (sharper pulses, stronger growth or coupling), as a new run; D4-1 stays closed |
| `adjoint − residual` CI < 0 | Co-state weighting beats goal-agnostic weighting with the same `τ̂` | Expected from theory; confirms the domain rewards goal-orientation |
| `adjoint − goal_local` CI < 0 | Propagation through the dynamics matters beyond projecting on the goal | The part that makes D4 useful for co-state research |
| `adjoint − goal_local` tie | Propagation adds nothing on this family | Report it; the family may be too weakly coupled |
| `exact_tau_adjoint − adjoint` large and negative | The step-doubling estimate misses error (e.g. under-resolved pulses) | Room for a better estimator; not a co-state failure |
| Equal-compute payoff FAIL (uniform ahead at equal compute) | Scoring one refinement at a time costs more than it saves on this family | D4-1 stays closed. Design a compute-efficient scoring scheme (mark many intervals per pass, incremental estimates, a learned amortised scorer) and validate it at equal compute before any learned-vs-critic comparison |
| Equal-compute payoff PASS | The adaptive policy pays for its own scoring | D4-1 may open |

Import with the `import-run` skill (everything under `config/`, `tests/`, `artifacts/`, `figures/`, `reports/` is small), then write the note with `research-note` and run `claim-check`.